# Module 02 — La comparaison

**Formation Big Data — ANSD / Data Innovation Lab**

Vous avez mesuré Polars, DuckDB et Dask sur votre machine et votre fichier.
Rassemblons tout, ajoutons pandas comme point de référence, et tirons-en une
grille de décision.

Ce notebook est court et surtout **interprétatif**. L'enjeu n'est pas de
désigner un vainqueur — il dépend de votre machine, de votre volume et de votre
usage — mais d'acquérir le réflexe : **mesurer avant de choisir**.

## 1. Contexte et rappel du protocole

In [ ]:
%load_ext autoreload 
%autoreload 2
import sys
from pathlib import Path
try:
    sys.path.append(str(Path(__file__).parent.parent.resolve()))
except NameError:
    sys.path.append(str(Path.cwd().parent.resolve()))
from tools.outils_mesure import (FICHIER, FICHIER_REGIONS, VOLUME_COMPARAISON,
                           afficher_protocole, contexte_machine, enregistrer,
                           memoire_mo, mesurer, charger_toutes, OPERATIONS)
import matplotlib.pyplot as plt
import pandas as pd

machine = contexte_machine()

In [ ]:
afficher_protocole()

## 2. Remesure de pandas

pandas a déjà été mesuré au module précédent, mais dans une autre session : le
cache du système et la charge de la machine n'étaient pas les mêmes. Pour que
la comparaison soit rigoureuse, nous le remesurons **ici**, dans les mêmes
conditions que les trois autres.

C'est aussi une bonne habitude : on ne compare que des mesures produites dans le
même contexte.

In [ ]:
# Fourni : campagne pandas sur le volume de comparaison
_ = pd.read_csv(FICHIER, nrows=10_000)          # lecture à blanc (cache)

df = pd.read_csv(FICHIER, nrows=VOLUME_COMPARAISON)
reference = df[["id_individu", "nom"]].sample(frac=0.5, random_state=1)

mesures_pandas = [
    mesurer("lecture",    lambda: pd.read_csv(FICHIER, nrows=VOLUME_COMPARAISON)),
    mesurer("filtre",     lambda: df[df["age"] >= 15]),
    mesurer("agregation", lambda: df.groupby("region")["age"].mean()),
    mesurer("tri",        lambda: df.sort_values(["region", "age"])),
    mesurer("jointure",   lambda: df.merge(reference, on="id_individu", how="left")),
]

enregistrer(mesures_pandas, outil="pandas", volume="2M",
            lignes=VOLUME_COMPARAISON)

In [ ]:
# Fourni : même campagne sur le fichier complet.
# C'est ici que pandas est censé rencontrer ses limites : les échecs seront
# enregistrés tels quels.
import gc

del df, reference
gc.collect()

mesures_pandas_completes = []
try:
    df_complet = pd.read_csv(FICHIER)
    reference_complete = df_complet[["id_individu", "nom"]].sample(
        frac=0.5, random_state=1)
    lignes_completes = len(df_complet)

    mesures_pandas_completes = [
        mesurer("lecture",    lambda: pd.read_csv(FICHIER)),
        mesurer("filtre",     lambda: df_complet[df_complet["age"] >= 15]),
        mesurer("agregation", lambda: df_complet.groupby("region")["age"].mean()),
        mesurer("tri",        lambda: df_complet.sort_values(["region", "age"])),
        mesurer("jointure",   lambda: df_complet.merge(
            reference_complete, on="id_individu", how="left")),
    ]
except MemoryError:
    print("pandas n'a pas pu charger le fichier complet — c'est un résultat.")
    lignes_completes = None
    mesures_pandas_completes = [
        {"operation": op, "secondes": None, "pic_memoire_mo": None,
         "surcout_memoire_mo": None, "statut": "échec",
         "motif": "chargement impossible"} for op in OPERATIONS
    ]

enregistrer(mesures_pandas_completes, outil="pandas", volume="complet",
            lignes=lignes_completes)

In [ ]:
# Libération avant l'analyse
for nom in ["df_complet", "reference_complete"]:
    if nom in dir():
        del globals()[nom]
gc.collect()

## 3. Rassemblement des mesures

In [ ]:
toutes = charger_toutes()
print(f"{len(toutes)} mesures, {toutes['outil'].nunique()} outils")
toutes.head(10)

In [ ]:
# Vérification : toutes les campagnes sont-elles présentes ?
toutes.groupby(["outil", "volume"]).size().unstack(fill_value=0)

## 4. Le comparatif, au volume de référence

In [ ]:
# Fourni — construisez un tableau croisé des temps :
#   une ligne par opération, une colonne par outil, au volume "2M",
#   en ne gardant que les mesures réussies (statut == "ok").
#
# Pour l'ordre des lignes : .reindex(OPERATIONS)

temps = toutes.query("volume == '2M' and statut == 'ok'") \
    .pivot(index="operation", columns="outil", values="secondes") \
    .reindex(OPERATIONS)
temps.round(3)

In [ ]:
# Fourni : graphique comparatif
axes = temps.plot.bar(figsize=(11, 4.5), rot=0)
axes.set_ylabel("secondes")
axes.set_title(f"Temps par opération — {VOLUME_COMPARAISON:,} lignes"
               .replace(",", " "))
axes.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fourni : même graphique en échelle logarithmique.
# Indispensable quand les écarts dépassent un facteur 10 : sinon les petites
# barres deviennent illisibles.
axes = temps.plot.bar(figsize=(11, 4.5), rot=0, logy=True)
axes.set_ylabel("secondes (échelle log)")
axes.set_title("Mêmes données, échelle logarithmique")
axes.grid(axis="y", alpha=0.3, which="both")
plt.tight_layout()
plt.show()

### Facteurs d'accélération

Rapporter chaque temps à celui de pandas rend les écarts immédiatement
lisibles.

In [ ]:
# Fourni — calculez, pour chaque opération et chaque outil, le facteur
# d'accélération par rapport à pandas.
#
# Un facteur de 4 signifie « quatre fois plus rapide que pandas ».
# Un facteur inférieur à 1 signifie « plus lent que pandas ».
#
# tableau — exactement ce qu'il faut ici.

acceleration = temps.rdiv(temps["pandas"], axis=0) \
    .drop(columns="pandas") \
    .reindex(OPERATIONS)
acceleration.round(1)

**Question 1.** Quel outil est le plus rapide sur votre machine, et sur
quelles opérations ? Y a-t-il des opérations où le classement s'inverse ?

*Votre réponse :* …

**Question 2.** Rapprochez ce résultat du nombre de cœurs de votre poste
(relevé en section 1). Un participant disposant de deux fois plus de cœurs
obtiendrait-il le même classement ?

*Votre réponse :* …

## 5. Et la mémoire ?

Le temps n'est pas le seul critère — et ce n'est même pas le principal, puisque
le mur du module précédent était un mur de mémoire.

In [ ]:
# Fourni : pic de mémoire par outil et par opération
memoire = (toutes.query("volume == '2M' and statut == 'ok'")
                 .pivot(index="operation", columns="outil",
                        values="surcout_memoire_mo")
                 .reindex(OPERATIONS))
memoire.round(0)

**Question 3.** Quel outil consomme le moins de mémoire supplémentaire ?
Comment expliquez-vous l'écart avec pandas ?

*Votre réponse :* …

> Attention à l'interprétation pour Dask : les processus de travail sont des
> processus **séparés**, donc leur mémoire n'apparaît pas dans celle de votre
> notebook. Le chiffre mesuré ici sous-estime la consommation réelle. C'est
> une limite de notre protocole, et il faut savoir la signaler.

## 6. Le fichier complet : qui tient encore debout ?

C'est le tableau le plus parlant du bloc.

In [ ]:
# Fourni : statut de chaque opération sur le fichier complet
statuts = (toutes.query("volume == 'complet'")
                 .pivot(index="operation", columns="outil", values="statut")
                 .reindex(OPERATIONS))
statuts

In [ ]:
# Fourni : temps sur le fichier complet, pour les opérations réussies
temps_complet = (toutes.query("volume == 'complet' and statut == 'ok'")
                       .pivot(index="operation", columns="outil",
                              values="secondes")
                       .reindex(OPERATIONS))
temps_complet.round(2)

In [ ]:
# Fourni : motifs des échecs, s'il y en a
echecs = toutes.query("statut == 'échec'")[["outil", "volume", "operation", "motif"]]
echecs if len(echecs) else print("Aucun échec : votre machine a tenu le choc.")

**Question 4.** Quelles cases sont vides ou en échec ? Ces échecs
concernent-ils plutôt certaines opérations que d'autres ? Pourquoi celles-là ?

*Votre réponse :* …

**Question 5.** Sur les opérations qui ont réussi partout, l'écart entre les
outils est-il plus grand ou plus petit qu'au volume de référence ? Qu'est-ce
que cela suggère ?

*Votre réponse :* …

## 7. Vitesse brute ou optimisation du plan ?

Rappel du protocole : jusqu'ici nous avons comparé les outils **à travail
identique** — chacun charge tout, puis calcule. Mesurons maintenant **à
objectif identique** : on demande seulement le résultat final, et chaque outil
optimise comme il l'entend.

In [ ]:
# Fourni : le même résultat, demandé de la façon la plus naturelle à chaque outil
import duckdb
import polars as pl

CHEMIN = str(FICHIER).replace("\\", "/")
con = duckdb.connect()
con.sql("SET enable_progress_bar = false")

print("À objectif identique — effectif et âge moyen par région :\n")

m_pandas = mesurer("pandas", lambda: (
    pd.read_csv(FICHIER, usecols=["region", "age"])
      .groupby("region")["age"].agg(["size", "mean"])))

m_polars = mesurer("polars", lambda: (
    pl.scan_csv(FICHIER)
      .group_by("region")
      .agg(pl.len(), pl.col("age").mean())
      .collect()))

m_duckdb = mesurer("duckdb", lambda: con.sql(
    f"SELECT region, COUNT(*), AVG(age) FROM '{CHEMIN}' GROUP BY 1").fetchall())

**Question 6.** Comparez ces temps à ceux de la ligne « agrégation » du
premier tableau. L'écart entre les outils s'est-il creusé ou resserré ?

*Votre réponse :* …

**Question 7.** Quelle part du gain vient de la **vitesse du moteur**, et
quelle part de sa capacité à **optimiser le plan** (ne lire que deux colonnes
sur vingt et une, filtrer pendant la lecture) ? Quelle conséquence pratique
pour la façon d'écrire vos traitements ?

*Votre réponse :* …

C'est la leçon la plus transférable du bloc : sur des données volumineuses,
**la façon de formuler le calcul pèse souvent plus lourd que le choix de
l'outil**.

## 8. Votre grille de décision

À vous de conclure. Complétez ce tableau en fonction de ce que vous avez
mesuré, de votre environnement de travail et des compétences de votre équipe.

| Situation | Mon choix | Pourquoi |
|---|---|---|
| Un fichier d'enquête de quelques dizaines de milliers de lignes | … | … |
| Un recensement régional, sur mon poste | … | … |
| Le RGPH complet, sur un serveur de la DMCI | … | … |
| Une analyse ponctuelle, que je veux écrire en SQL | … | … |
| Un traitement récurrent qui doit passer à l'échelle plus tard | … | … |
| Des données qui dépassent ce qu'une seule machine peut contenir | … | … |

**Question 8.** Après ce bloc, recommanderiez-vous au Data Innovation Lab de
monter un cluster Spark pour traiter le RGPH ? Argumentez avec vos chiffres.

*Votre réponse :* …

## 9. Ce qu'il faut retenir

- **Aucun outil ne gagne partout.** Le classement dépend de l'opération, du
  volume, du nombre de cœurs et de la mémoire disponible.
- Les moteurs modernes sur **une seule machine** (Polars, DuckDB) couvrent la
  très grande majorité des besoins d'un institut de statistique.
- **Dask** ne cherche pas à gagner ici : il prépare le passage à plusieurs
  machines, et son modèle d'exécution est celui de Spark.
- La **façon d'écrire le calcul** — laisser le moteur optimiser plutôt que tout
  charger — pèse souvent plus lourd que le choix de l'outil.
- Le réflexe à conserver : **mesurer sur ses propres données, avant de
  choisir**. Vous savez maintenant le faire.

**Une question reste ouverte.** Toutes ces mesures ont été faites sur un fichier
**CSV** : un format texte qu'il faut analyser, ligne par ligne, colonne par
colonne, à chaque lecture. Vous avez d'ailleurs vu avec DuckDB que ce coût
d'analyse est considérable.

Et si le format lui-même était le problème ?


In [ ]:
# Sauvegarde du comparatif, pour le rapport de formation
temps.to_csv("resultats/comparatif_temps_2M.csv")
print("Comparatif enregistré.")